# 🏗️ Rapport de Data Engineering & KPI Confort Urbain
Ce notebook regroupe et documente les différents scripts utilisés pour construire l'architecture de données de notre projet (Pipeline **Bronze ➔ Silver ➔ Gold**).


## 1. Quartiers de Paris 🗺️
Voici le script utilisé pour préparer les données de `silver_quartiers.py`.


In [ ]:
import pandas as pd
import os

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'commun')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'confort')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_quartiers():
    print("Démarrage du pipeline Silver : Quartiers de Paris...")

    file_path = os.path.join(BRONZE_DIR, 'quartier_paris.csv')

    df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', on_bad_lines='skip')

    print(f"  Colonnes dispo: {df.columns.tolist()}")
    print(f"  Lignes lues   : {len(df)}")

    # Renommage propre
    rename_map = {
        'N_SQ_QU':       'id_quartier',
        'C_QU':          'code_quartier',
        'C_QUINSEE':     'code_insee_quartier',
        'L_QU':          'nom_quartier',
        'C_AR':          'arrondissement',
        'SURFACE':       'surface_m2',
        'PERIMETRE':     'perimetre',
        'Geometry X Y':  'centroide_xy',
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # Nettoyage
    if 'arrondissement' in df.columns:
        df['arrondissement'] = pd.to_numeric(df['arrondissement'], errors='coerce')

    # On garde les colonnes essentielles
    keep = [c for c in ['id_quartier', 'code_quartier', 'code_insee_quartier', 'nom_quartier',
                         'arrondissement', 'surface_m2', 'perimetre', 'centroide_xy'] if c in df.columns]
    df_silver = df[keep].copy()

    print(f"  Aperçu :\n{df_silver.head(3).to_string()}")

    os.makedirs(SILVER_DIR, exist_ok=True)
    df_silver.to_csv(os.path.join(SILVER_DIR, 'silver_quartiers.csv'), index=False, sep=';', encoding='utf-8-sig')

    print("✓ Table silver_quartiers générée.")

if __name__ == "__main__":
    process_silver_quartiers()



Démarrage du pipeline Silver : Quartiers de Paris...
  Colonnes dispo: ['N_SQ_QU', 'C_QU', 'C_QUINSEE', 'L_QU', 'C_AR', 'N_SQ_AR', 'PERIMETRE', 'SURFACE', 'Geometry X Y', 'Geometry', 'st_area(shape)', 'st_perimeter(shape)']
  Lignes lues   : 80
  Aperçu :
   id_quartier  code_quartier  code_insee_quartier           nom_quartier  arrondissement    surface_m2    perimetre                            centroide_xy
0    750000022             22              7510602                  Odéon               6  7.161484e+05  3516.314464  48.847800629292664, 2.3363388275876775
1    750000043             43              7511103               Roquette              11  1.172087e+06  4973.010557    48.85706404083103, 2.380364061726767
2    750000023             23              7510603  Notre-Dame-des-Champs               6  8.613070e+05  4559.989773    48.84642759400415, 2.327356878234607
✓ Table silver_quartiers générée.


## 2. Gares et Transports 🚆
Voici le script utilisé pour préparer les données de `silver_gares.py`.


In [ ]:
import pandas as pd
import sqlite3
import os
import ast

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'confort')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'confort')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_gares():
    print("Démarrage du pipeline Silver : Gares de Paris...")

    file_path = os.path.join(BRONZE_DIR, 'emplacement-des-gares-idf.csv')

    df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', on_bad_lines='skip')

    print(f"  Lignes lues au départ : {len(df)}")

    # Extraire les coordonnées à partir de "Geo Point" (format: lat, lon)
    if 'Geo Point' in df.columns:
        df[['latitude', 'longitude']] = df['Geo Point'].str.split(',', expand=True).astype(float)
    
    # Stratégie de filtrage : Paris intra-muros
    # Limites approximatives de Paris (WGS84)
    # Latitude: 48.815 à 48.905
    # Longitude: 2.220 à 2.475
    
    # On ajoute aussi un filtre sur le nom pour récupérer les gares dont le nom contient Paris
    # et on exclut les cas aberrants
    mask_coords = (
        (df['latitude'] >= 48.815) & (df['latitude'] <= 48.905) &
        (df['longitude'] >= 2.220) & (df['longitude'] <= 2.475)
    )
    
    # On force quelques inclusions ou exclusions si besoin
    mask_name = df['nom_long'].str.contains('Paris', case=False, na=False) | df['nom_ZdC'].str.contains('Paris', case=False, na=False)
    
    df_paris = df[mask_coords | mask_name].copy()
    
    # Un nettoyage supplémentaire : parfois "Paris" est dans le nom mais c'est hors Paris (ex: Massy Palaiseau TGV)
    # Vérifions si les coordonnées sont complètement aberrantes
    mask_strict = (
        (df_paris['latitude'] >= 48.75) & (df_paris['latitude'] <= 48.95) &
        (df_paris['longitude'] >= 2.15) & (df_paris['longitude'] <= 2.55)
    )
    df_paris = df_paris[mask_strict]

    print(f"  Lignes conservées (Paris) : {len(df_paris)}")

    # Renommage propre
    rename_map = {
        'gares_id':       'id_gare',
        'nom_long':       'nom_gare',
        'nom_su_gar':     'nom_supplementaire',
        'res_com':        'reseau',
        'indice_lig':     'ligne',
        'mode':           'mode',
        'exploitant':     'exploitant',
        'latitude':       'latitude',
        'longitude':      'longitude',
        'nom_ZdC':        'zone_commerciale'
    }
    
    df_silver = df_paris.rename(columns={k: v for k, v in rename_map.items() if k in df_paris.columns})

    # On garde les colonnes essentielles
    keep = ['id_gare', 'nom_gare', 'nom_supplementaire', 'zone_commerciale', 'reseau', 'ligne', 'mode', 'exploitant', 'latitude', 'longitude']
    df_silver = df_silver[[c for c in keep if c in df_silver.columns]]

    # Nettoyage des chaînes
    for col in df_silver.select_dtypes(include=['object']).columns:
        df_silver[col] = df_silver[col].str.strip()

    print(f"  Aperçu :\n{df_silver.head(3).to_string()}")

    os.makedirs(SILVER_DIR, exist_ok=True)
    df_silver.to_csv(os.path.join(SILVER_DIR, 'silver_gares.csv'), index=False, sep=';', encoding='utf-8-sig')

    print("OK - Table silver_gares générée pour Paris.")

if __name__ == "__main__":
    process_silver_gares()



Démarrage du pipeline Silver : Gares de Paris...
  Lignes lues au départ : 1240
  Lignes conservées (Paris) : 524
  Aperçu :
    id_gare            nom_gare nom_supplementaire    zone_commerciale reseau ligne mode exploitant   latitude  longitude
25      574       Musée d'Orsay                NaN       Musée d'Orsay  RER C     C  RER       SNCF  48.860351   2.326760
26      832          Saint-Ouen                NaN          Saint-Ouen  RER C     C  RER       SNCF  48.904566   2.322626
37      189  Cité Universitaire                NaN  Cité universitaire  RER B     B  RER       RATP  48.821090   2.338978
OK - Table silver_gares générée pour Paris.


C:\Users\rayan\AppData\Local\Temp\ipykernel_11684\212465867.py:82: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_silver.select_dtypes(include=['object']).columns:


## 3. Loyers et Prix 💶
Voici le script utilisé pour préparer les données de `silver_prix.py`.


In [ ]:
import pandas as pd
import sqlite3
import os

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'immobilier')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'immobilier')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_prix():
    print("Démarrage du pipeline Silver : Prix & Loyers...")

    file_path = os.path.join(BRONZE_DIR, 'logement-encadrement-des-loyers.csv')

    # Lecture avec encodage UTF-8-sig (gère le BOM des exports Windows)
    df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', on_bad_lines='skip')

    print(f"  Colonnes dispo: {df.columns.tolist()}")
    print(f"  Lignes lues   : {len(df)}")

    # Sélection des colonnes utiles (par position si les accents posent problème)
    # Détection souple par mots-clés dans le nom de colonne
    col_map = {
        'annee':              next((c for c in df.columns if 'ann' in c.lower()), None),
        'secteur':            next((c for c in df.columns if 'secteur' in c.lower()), None),
        'num_quartier':       next((c for c in df.columns if 'num' in c.lower() and 'quartier' in c.lower()), None),
        'nom_quartier':       next((c for c in df.columns if 'nom' in c.lower() and 'quartier' in c.lower()), None),
        'nb_pieces':          next((c for c in df.columns if 'pi' in c.lower() and 'ces' in c.lower()), None),
        'epoque_construction':next((c for c in df.columns if 'poque' in c.lower()), None),
        'type_location':      next((c for c in df.columns if 'location' in c.lower()), None),
        'loyer_ref':          next((c for c in df.columns if 'r' in c.lower() and 'f' in c.lower() and 'rence' in c.lower() and 'major' not in c.lower() and 'minor' not in c.lower()), None),
        'loyer_majore':       next((c for c in df.columns if 'major' in c.lower()), None),
        'loyer_minore':       next((c for c in df.columns if 'minor' in c.lower()), None),
        'ville':              next((c for c in df.columns if c.lower() == 'ville'), None),
        'code_insee_quartier':next((c for c in df.columns if 'insee' in c.lower()), None),
    }

    # Construire le DataFrame Silver avec des noms normalisés
    rows = {alias: df[col] for alias, col in col_map.items() if col is not None}
    df_silver = pd.DataFrame(rows)

    # Extraire l'arrondissement depuis le code INSEE (ex: "7510102" -> arrondissement "1")
    if 'code_insee_quartier' in df_silver.columns:
        # Code INSEE format 75101xx -> les 5 premiers chiffres = 75101 (1er arr.)
        df_silver['code_arrondissement'] = df_silver['code_insee_quartier'].astype(str).str[:5]
        # Numéro d'arrondissement 1-20
        df_silver['arrondissement'] = df_silver['code_arrondissement'].astype(str).str[-2:].astype(int)

    # Conversion numérique des loyers
    for col in ['loyer_ref', 'loyer_majore', 'loyer_minore']:
        if col in df_silver.columns:
            df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

    # Export
    os.makedirs(SILVER_DIR, exist_ok=True)
    df_silver.to_csv(os.path.join(SILVER_DIR, 'silver_prix.csv'), index=False, sep=';', encoding='utf-8-sig')

    print(f"  Colonnes Silver : {df_silver.columns.tolist()}")
    print(f"  Aperçu :\n{df_silver.head(3).to_string()}")
    print("✓ CSV silver_prix généré.")

if __name__ == "__main__":
    process_silver_prix()



Démarrage du pipeline Silver : Prix & Loyers...
  Colonnes dispo: ['Année', 'Secteurs géographiques', 'Numéro du quartier', 'Nom du quartier', 'Nombre de pièces principales', 'Epoque de construction', 'Type de location', 'Loyers de référence', 'Loyers de référence majorés', 'Loyers de référence minorés', 'Ville', 'Numéro INSEE du quartier', 'geo_shape', 'geo_point_2d']
  Lignes lues   : 17920
  Colonnes Silver : ['annee', 'secteur', 'num_quartier', 'nom_quartier', 'nb_pieces', 'epoque_construction', 'type_location', 'loyer_ref', 'loyer_majore', 'loyer_minore', 'ville', 'code_insee_quartier', 'code_arrondissement', 'arrondissement']
  Aperçu :
   annee  secteur  num_quartier  nom_quartier  nb_pieces epoque_construction type_location  loyer_ref  loyer_majore  loyer_minore  ville  code_insee_quartier code_arrondissement  arrondissement
0   2023       11            77    Belleville          3           1946-1970    non meublé       19.8          23.8          13.9  PARIS              75120

## 4. Qualité Urbaine (Incivilités) 🗑️
Voici le script utilisé pour préparer les données de `silver_qualite.py`.


In [ ]:
import pandas as pd
import sqlite3
import os

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'confort')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'confort')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_qualite():
    print("Démarrage du pipeline Silver : Qualité Urbaine (dans-ma-rue)...")

    file_path = os.path.join(BRONZE_DIR, 'dans-ma-rue.csv')

    # Colonnes utiles uniquement pour économiser la RAM sur ce gros fichier
    cols_to_use = ['TYPE DECLARATION', 'SOUS TYPE DECLARATION', 'ARRONDISSEMENT',
                   'CODE POSTAL', 'ANNEE DECLARATION', 'MOIS DECLARATION', 'INTERVENANT']

    try:
        df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig',
                         usecols=cols_to_use, on_bad_lines='skip')
    except Exception:
        # Fallback si l'encodage diffère
        df = pd.read_csv(file_path, sep=';', encoding='latin-1',
                         usecols=cols_to_use, on_bad_lines='skip')

    print(f"  Lignes lues   : {len(df)}")
    print(f"  Colonnes dispo: {df.columns.tolist()}")

    # Nettoyage et renommage
    df = df.rename(columns={
        'TYPE DECLARATION':      'type_declaration',
        'SOUS TYPE DECLARATION': 'sous_type_declaration',
        'ARRONDISSEMENT':        'arrondissement',
        'CODE POSTAL':           'code_postal',
        'ANNEE DECLARATION':     'annee',
        'MOIS DECLARATION':      'mois',
        'INTERVENANT':           'intervenant',
    })

    # Nettoyage de l'arrondissement (on veut 1 à 20)
    df['arrondissement'] = pd.to_numeric(df['arrondissement'], errors='coerce')
    df = df.dropna(subset=['arrondissement'])
    df['arrondissement'] = df['arrondissement'].astype(int)

    # Agrégation : nombre de signalements par arrondissement / type / année
    df_groupe = df.groupby(
        ['arrondissement', 'type_declaration', 'annee'], dropna=True
    ).size().reset_index(name='nb_signalements')

    print(f"  Lignes Silver : {len(df_groupe)}")
    print(f"  Aperçu :\n{df_groupe.head(5).to_string()}")

    os.makedirs(SILVER_DIR, exist_ok=True)
    df_groupe.to_csv(os.path.join(SILVER_DIR, 'silver_qualite_urbaine.csv'), index=False, sep=';', encoding='utf-8-sig')

    print("✓ CSV silver_qualite_urbaine généré.")

if __name__ == "__main__":
    process_silver_qualite()



Démarrage du pipeline Silver : Qualité Urbaine (dans-ma-rue)...
  Lignes lues   : 1062299
  Colonnes dispo: ['TYPE DECLARATION', 'SOUS TYPE DECLARATION', 'CODE POSTAL', 'ARRONDISSEMENT', 'ANNEE DECLARATION', 'MOIS DECLARATION', 'INTERVENANT']
  Lignes Silver : 334
  Aperçu :
   arrondissement                            type_declaration  annee  nb_signalements
0               0  Activités commerciales et professionnelles   2025                1
1               0        Autos, motos, vélos, trottinettes...   2025                6
2               0   Graffitis, tags, affiches et autocollants   2025               22
3               0                           Objets abandonnés   2025               32
4               0                                    Propreté   2025               13
✓ Table silver_qualite_urbaine générée.


## 5. Démographie 👥
Voici le script utilisé pour préparer les données de `silver_demographie.py`.


In [ ]:
import pandas as pd
import sqlite3
import os

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'commun')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'commun')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_demographie():
    print("Démarrage du pipeline Silver : Démographie (IRIS)...")

    file_path = os.path.join(BRONZE_DIR, 'iris.csv')

    # Le fichier est séparé par des virgules avec encodage UTF-8
    df = pd.read_csv(file_path, sep=',', encoding='utf-8-sig', on_bad_lines='skip')

    print(f"  Colonnes disponibles (5 premières): {df.columns[:5].tolist()}")
    print(f"  Nombre de lignes: {len(df)}")

    # Détection souple des colonnes clés
    cols_map = {
        'code_iris':  next((c for c in df.columns if 'c_iris' in c.lower() or ('cainsee' in c.lower()) or ('iris' in c.lower() and 'code' in c.lower())), None),
        'nom_iris':   next((c for c in df.columns if 'l_iris' in c.lower() or ('iris' in c.lower() and 'nom' in c.lower())), None),
        'population': next((c for c in df.columns if 'nb_pop' in c.lower() or ('pop' in c.lower() and 'densite' not in c.lower())), None),
        'densite':    next((c for c in df.columns if 'densite' in c.lower()), None),
        'surface_ha': next((c for c in df.columns if 'surf_ha' in c.lower() or ('surface' in c.lower() and 'ha' in c.lower())), None),
    }

    print(f"  Mapping colonnes détecté: {cols_map}")

    # Construire le DataFrame Silver
    rows = {alias: df[col] for alias, col in cols_map.items() if col is not None}
    df_silver = pd.DataFrame(rows)

    # Extraire l'arrondissement depuis le code INSEE IRIS (ex: "750010118" -> "75001" -> arrondissement 1)
    if 'code_iris' in df_silver.columns:
        df_silver['code_arrondissement'] = df_silver['code_iris'].astype(str).str[:5]
        df_silver['arrondissement'] = df_silver['code_arrondissement'].astype(str).str[-2:].apply(
            lambda x: int(x) if x.isdigit() else None
        )

    # Conversion numérique
    for col in ['population', 'densite', 'surface_ha']:
        if col in df_silver.columns:
            df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

    # Agrégation par arrondissement
    if 'arrondissement' in df_silver.columns and 'population' in df_silver.columns:
        df_groupe = df_silver.groupby('arrondissement').agg(
            population_totale=('population', 'sum'),
            nb_iris=('code_iris', 'count')
        ).reset_index()
    else:
        df_groupe = df_silver.copy()

    print(f"  Lignes Silver : {len(df_groupe)}")
    print(f"  Aperçu :\n{df_groupe.head(5).to_string()}")

    os.makedirs(SILVER_DIR, exist_ok=True)
    df_groupe.to_csv(os.path.join(SILVER_DIR, 'silver_demographie.csv'), index=False, sep=';', encoding='utf-8-sig')

    print("✓ CSV silver_demographie généré.")

if __name__ == "__main__":
    process_silver_demographie()



Démarrage du pipeline Silver : Démographie (IRIS)...
  Colonnes disponibles (5 premières): ['Geo Shape;DEP;INSEE_COM;NOM_COM;IRIS;CODE_IRIS;NOM_IRIS;TYP_IRIS;Geo Point;ID']
  Nombre de lignes: 992
  Mapping colonnes détecté: {'code_iris': 'Geo Shape;DEP;INSEE_COM;NOM_COM;IRIS;CODE_IRIS;NOM_IRIS;TYP_IRIS;Geo Point;ID', 'nom_iris': 'Geo Shape;DEP;INSEE_COM;NOM_COM;IRIS;CODE_IRIS;NOM_IRIS;TYP_IRIS;Geo Point;ID', 'population': None, 'densite': None, 'surface_ha': None}
  Lignes Silver : 992
  Aperçu :
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## 6. Travaux d'Aménagement 🚧
Voici le script utilisé pour préparer les données de `silver_travaux.py`.


In [ ]:
import pandas as pd
import sqlite3
import os

def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")

BASE_DIR = resolve_base_dir()
BRONZE_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'confort')
SILVER_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'confort')
GOLD_DIR   = os.path.join(BASE_DIR, 'data', 'gold')

def process_silver_travaux():
    print("Démarrage du pipeline Silver : Travaux Paris (parissetransforme)...")

    file_path = os.path.join(BRONZE_DIR, 'parissetransforme.csv')

    try:
        df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', on_bad_lines='skip')
    except Exception:
        df = pd.read_csv(file_path, sep=';', encoding='latin-1', on_bad_lines='skip')

    print(f"  Colonnes dispo: {df.columns.tolist()}")
    print(f"  Lignes lues   : {len(df)}")

    # Renommage souple par mots-clés
    rename_map = {}
    for c in df.columns:
        cl = c.lower()
        if 'titre' in cl or 'title' in cl:
            rename_map[c] = 'titre_projet'
        elif 'sous' in cl and 'cat' in cl:
            rename_map[c] = 'sous_categorie'
        elif 'cat' in cl:
            rename_map[c] = 'categorie'
        elif 'adresse' in cl:
            rename_map[c] = 'adresse'
        elif 'postal' in cl or 'cp' in cl:
            rename_map[c] = 'code_postal'
        elif 'livraison' in cl or 'date' in cl:
            rename_map[c] = 'date_livraison'

    df = df.rename(columns=rename_map)

    # Extraire l'arrondissement à partir du code postal (75001 -> 1, 75020 -> 20)
    if 'code_postal' in df.columns:
        cp = df['code_postal'].astype(str).str.strip()
        cp = cp.str.replace(r'\.0$', '', regex=True)
        arr = cp.str.extract(r'(\d{2})$')[0]
        df['arrondissement'] = pd.to_numeric(arr, errors='coerce').astype('Int64')

    # Sélectionner colonnes finales
    keep = [c for c in ['titre_projet', 'categorie', 'sous_categorie',
                         'adresse', 'code_postal', 'arrondissement', 'date_livraison'] if c in df.columns]
    df_silver = df[keep].drop_duplicates()

    print(f"  Lignes Silver : {len(df_silver)}")
    print(f"  Aperçu :\n{df_silver.head(3).to_string()}")

    os.makedirs(SILVER_DIR, exist_ok=True)
    df_silver.to_csv(os.path.join(SILVER_DIR, 'silver_travaux.csv'), index=False, sep=';', encoding='utf-8-sig')

    print("✓ CSV silver_travaux généré.")

if __name__ == "__main__":
    process_silver_travaux()



Démarrage du pipeline Silver : Travaux Paris (parissetransforme)...
  Colonnes dispo: ['Titre du descriptif', 'Corps du descriptif', 'Catégorie', 'Sous-Catégorie', 'Adresse', 'Code postal', 'Date de livraison estimée', 'URL page paris.fr', 'URL Photo', 'Crédit photo', 'geo_shape', 'geo_point_2d']
  Lignes lues   : 3262
  Lignes Silver : 3257
  Aperçu :
                                    titre_projet               categorie                                      sous_categorie                                 adresse  code_postal  arrondissement date_livraison
0              Epicerie circuit court -  M.I.A.M  Attractivité et emploi  Aide à l’installation de commerçants et d’artisans  52, Rue Curial - 13, Passage De Crimee      75019.0              19            NaN
1  Une nouvelle station Trilib près de chez vous                Propreté                                     Stations Trilib                       39 Ter Gay Lussac      75005.0               5            NaN
2             Renc

# 🏆 Modélisation Gold : KPI de Confort Urbain

Cette section présente le calcul du score final basé sur les couches Silver croisées. Voici le détail de la méthodologie retenue :


# KPI de Confort Urbain — Paris, 80 Quartiers
## Documentation Technique du Pipeline Gold

**Sources de données :** `dans-ma-rue.csv` · `parissetransforme.csv` · `logement-encadrement-des-loyers.csv` · `quartier_paris.csv` · `silver_gares.csv`
**Granularité :** 1 ligne = 1 quartier parmi les 80 quartiers administratifs de Paris
**Loyer médian parisien observé :** 29,15 €/m² (2025, non meublé, 2 pièces)

---

## Données et notations

Pour chaque quartier **q** parmi les **80 quartiers de Paris** (référentiel `quartier_paris.csv`) :

| Notation | Description | Source |
|---|---|---|
| **S(q)** | Surface du quartier en m² (colonne `SURFACE`) | `quartier_paris.csv` |
| **N_prop(q)** | Nombre de signalements Propreté/Voirie dans q | `dans-ma-rue.csv` |
| **N_incivil(q)** | Somme pondérée des signalements Incivilités dans q | `dans-ma-rue.csv` |
| **N_projets(q)** | Somme pondérée des projets d'aménagement dans q | `parissetransforme.csv` |
| **L(q)** | Loyer moyen de référence au m² dans q (2025, non meublé, 2 pièces) | `logement-encadrement-des-loyers.csv` |
| **G(q)** | Nombre de gares (métro, RER, tram) situées à moins de 800m du centre du quartier | `silver_gares.csv` |
| **rang(x)** | Rang centile de x parmi les 80 quartiers → valeur entre 0 et 1 | Calculé par `pandas.rank(pct=True)` |

> **Jointure géographique :** Les signalements `dans-ma-rue` sont rattachés aux quartiers via la colonne `CONSEIL DE QUARTIER` (matching sur nom normalisé ↔ `L_QU` du référentiel). Les loyers sont joints via le code INSEE du quartier (`Numéro INSEE du quartier` = `C_QUINSEE`). Les projets `parissetransforme` sont joints via le code postal (`75001` → arrondissement `1`) puis distribués uniformément sur les 4 quartiers de l'arrondissement.

---

## S1 — Score de Propreté

**Source :** `dans-ma-rue.csv`
**Types de signalements retenus :** `"Propreté"` + `"Voirie et espace public"`

```
              N_prop(q)
d_prop(q) = ─────────────       (densité de signalements / m²)
                S(q)

S1(q) = 10 * (1 - rang(d_prop(q)))
```

> Moins il y a de signalements rapportés à la surface → score plus élevé

**Valeurs observées (extrait) :**
- Quartiers ruraux/résidentiels (16ème, 7ème) : d_prop faible → S1 élevé
- Quartiers très denses et touristiques (9ème, 1er) : d_prop fort → S1 faible

---

## S2 — Score d'Incivilités

**Source :** `dans-ma-rue.csv`
**Types retenus :** `"Objets abandonnés"` + `"Graffitis, tags, affiches et autocollants"` + `"Mobiliers urbains"`

Chaque signalement reçoit un poids de gravité **w** selon son sous-type :

```
Sous-type (détecté par mots-clés dans SOUS TYPE DECLARATION)  w
────────────────────────────────────────────────────────────────
Objets infestés (punaises de lit)                             3.0
Épanchement d'urine                                           2.0
Graffitis sur mur / façade / rideau métallique                1.5
Déjections canines                                            1.5
Graffitis (autres)                                            1.3
Objets encombrants, cartons, gravats                          1.0
Autres (défaut)                                               1.0
```

```
I_incivil(q)  =  w(s1) + w(s2) + ... + w(sn)
                 (somme des poids de tous les signalements dans q)

                  I_incivil(q)
d_incivil(q)  =  ─────────────
                     S(q)

S2(q)  =  10 * (1 - rang(d_incivil(q)))
```

> Moins la pression d'incivilité pondérée est forte → score plus élevé

**Effet multiplicatif observé :** Belleville (20ème) présente `S2 ≈ 0.01`, ce qui ramène son KPI final à `0.27` malgré un excellent S3 = 10.0.

---

## S3 — Score d'Aménagement Urbain

**Source :** `parissetransforme.csv`
**Volume :** 3 262 projets référencés (dont 3 257 uniques après déduplication)

Chaque projet reçoit un poids **w** selon sa catégorie (`Catégorie`) :

```
Catégorie (colonne "Catégorie")                w     Nb projets observés
───────────────────────────────────────────────────────────────────────
Espace public / Nature en ville               2.0         408
Écoles et crèches                             1.8         333
Propreté (Stations Trilib, collecte…)         1.5         502
Sécurité et Prévention                        1.3          39
Sport                                         1.0          90
Autres catégories (Attractivité, Culture…)    0.5       1 890
```

```
I_am(q)   =  w(p1) + w(p2) + ... + w(pm)
             (somme des poids de tous les projets dans les 4 quartiers de l'arrondissement,
              répartis uniformément car les projets sont géolocalisés au niveau arrondissement)

              I_am(q)
d_am(q)  =  ──────────
               S(q)

S3(q)  =  10 * rang(d_am(q))
```

> Plus il y a de projets d'aménagement rapportés à la surface → score plus élevé

**Note :** Les arrondissements les plus denses en projets sont le 20ème (337 projets) et le 19ème (299 projets). Les plus aisés (16ème, 7ème) sont en queue de classement sur cet indicateur.

---

## S4 — Score d'Accessibilité du Logement

**Source :** `logement-encadrement-des-loyers.csv`
**Filtre :** Année 2025, type `"non meublé"`, 2 pièces principales, toutes époques de construction.
**Volume utilisé :** ~17 920 lignes de loyers encadrés par Paris.

```
             L(q, Avant 1946) + L(q, 1946-1970) + L(q, 1971-1990) + L(q, Après 1990)
L_moy(q)  =  ────────────────────────────────────────────────────────────────────────
                                              4

L_med  =  médiane de { L_moy(q) pour tous les q } = 29,15 €/m²   (médiane parisienne 2025)

             | L_moy(q) - L_med |
delta(q)  =  ────────────────────
                    L_med

                        delta(q)
S4(q)  =  10 * (1  -  ─────────────────────────)
                        max( delta(q') , q' ∈ Q )
```

> Le score est maximal quand le loyer est proche de la médiane parisienne (29,15 €/m²).
> Un loyer très haut (quartiers chics, inaccessibles) **ou** très bas (quartiers dégradés) → score faible.

**Imputation :** Les quartiers sans données de loyer directes sont imputés par la médiane de leur arrondissement.

---

---

## S5 — Score de Connectivité Transport

**Source :** `silver_gares.csv` (filtré depuis la donnée régionale)
**Méthodologie :** Rayon d'accessibilité piétonne locale (10 min de marche).

Pour chaque quartier, on calcule la distance à vol d'oiseau entre le centre du quartier (centroïde) et l'ensemble des gares (SNCF / RATP / Tram) de Paris. 

```
G(q) = Nombre de stations situées à moins de 0.8 km (800 m) du centre de q

S5(q)  =  10 * rang( G(q) )
```

> Plus un quartier est entouré de stations de transport lourd à courte distance, plus son score grimpe.

**Valeurs observées :**
- L'hypercentre présente logiquement des densités record de stations de métro dans un rayon de 800m.
- Les grands quartiers excentrés sont défavorisés sur ce critère très local.

---

## KPI Final — Indice de Confort Urbain

### Formule

```
KPI(q)  =  S1(q)^0.30  *  S2(q)^0.25  *  S3(q)^0.15  *  S4(q)^0.15  *  S5(q)^0.15
```

### Poids et justification

```
Composante       Exposant   Justification métier
────────────────────────────────────────────────────────────────────────────
S1  Propreté       0.30     Critère n°1 de satisfaction des habitants parisiens
S2  Incivilités    0.25     Impact direct sur la sécurité perçue et la qualité de vie
S3  Aménagement    0.15     Indicateur prospectif : investissements de la Ville
S4  Loyer          0.15     Dimension économique : accessibilité au logement
S5  Transport      0.15     Dimension mobilités : accessibilité à pied (<800m)
────────────────────────────────────────────────────────────────────────────
Somme              1.00
```

### Pourquoi une moyenne géométrique (et non additive) ?

```
Exemple :        S1    S2    S3    S4    Somme pondérée    KPI (géométrique)
─────────────────────────────────────────────────────────────────────────────
Quartier A       10     0    10    10         7.0  ← trop haut        ≈ 0  ✓
Quartier B        6     6     6     6         6.0                      6.0
```

Un quartier avec **S2 = 0** (incivilités catastrophiques) obtient un **KPI ≈ 0**,
même s'il est parfait sur les 3 autres axes.

→ C'est le même principe que l'**Indice de Développement Humain (IDH) de l'ONU** :
**un seul axe catastrophique suffit à plomber le score global.**

### Plage du KPI

```
0  ──────────────────────────────────────────────  10
Très inconfortable                     Très confortable
```

---

## Résultats Observés — Classement des 80 Quartiers

### Top 10 (Quartiers les plus confortables)

| Rang | Quartier | Arr. | S1_proprete | S2_incivilites | S3_amenagement | S4_accessibilite | S5_transport | KPI_confort_urbain |
|---|---|---|---|---|---|---|---|---|
| 1 | Porte-Saint-Denis | 10 | 5.94 | 5.94 | 9.75 | 10.00 | 8.56 | 7.31 |
| 2 | Sainte-Avoie | 3 | 5.94 | 5.94 | 8.75 | 10.00 | 9.44 | 7.29 |
| 3 | Porte-Saint-Martin | 10 | 5.94 | 5.94 | 9.12 | 10.00 | 8.25 | 7.19 |
| 4 | Folie-Méricourt | 11 | 5.94 | 5.94 | 9.62 | 10.00 | 7.69 | 7.18 |
| 5 | Saint-Merri | 4 | 5.94 | 5.94 | 8.62 | 10.00 | 8.25 | 7.13 |
| 6 | Sainte-Marguerite | 11 | 5.94 | 5.94 | 9.00 | 10.00 | 6.94 | 7.00 |
| 7 | Chaussée-d'Antin | 9 | 5.94 | 5.94 | 6.12 | 10.00 | 10.00 | 6.97 |
| 8 | Gaillon | 2 | 5.94 | 5.94 | 6.62 | 10.00 | 8.88 | 6.93 |
| 9 | Notre-Dame | 4 | 5.94 | 5.94 | 7.50 | 10.00 | 7.69 | 6.91 |
| 10 | Arts-et-Métiers | 3 | 5.94 | 5.94 | 6.00 | 10.00 | 9.44 | 6.89 |

### Bottom 10 (Quartiers les moins confortables)

| Rang | Quartier | Arr. | S1_proprete | S2_incivilites | S3_amenagement | S4_accessibilite | S5_transport | KPI_confort_urbain | Axe pénalisant principal |
|---|---|---|---|---|---|---|---|---|---|
| 71 | Monnaie | 6 | 0.75 | 0.37 | 9.38 | 10.00 | 8.25 | 1.95 | Faible propreté/investissements locaux ou loyer |
| 72 | Halles | 1 | 5.94 | 5.94 | 1.12 | 0.01 | 9.44 | 1.90 | Accessibilité loyer critique |
| 73 | Chaillot | 16 | 0.88 | 0.88 | 1.88 | 10.00 | 6.25 | 1.90 | S1 / S2 / S3 très faibles |
| 74 | Jardin-des-Plantes | 5 | 0.62 | 0.62 | 4.12 | 10.00 | 5.44 | 1.74 | S1 / S2 très faibles |
| 75 | Porte-Dauphine | 16 | 1.75 | 1.37 | 0.62 | 10.00 | 0.94 | 1.67 | S1 / S2 / S3 / S5 très faibles (excentré) |
| 76 | Val-de-Grace | 5 | 0.50 | 0.50 | 5.50 | 10.00 | 0.94 | 1.23 | Transport et propreté perçue très bas |
| 77 | Sorbonne | 5 | 0.25 | 0.25 | 8.88 | 10.00 | 6.25 | 1.20 | Propreté et incivilités extrêmes (S1/S2) |
| 78 | Gros-Caillou | 7 | 0.37 | 1.00 | 0.75 | 10.00 | 0.69 | 0.95 | Propreté, aménagement urbain, transport (excentré) |
| 79 | Faubourg-Montmartre | 9 | 0.01 | 0.12 | 8.12 | 10.00 | 8.00 | 0.40 | Propreté et incivilités virtuellement nuls |
| 80 | Belleville | 20 | 0.12 | 0.01 | 10.00 | 10.00 | 2.25 | 0.38 | Incivilités écrasantes effondrant le KPI |

---

## Limites et Pistes d'Amélioration

### Limites connues
- **Jointure sur nom de quartier** : seulement 10,5% des signalements `dans-ma-rue` ont pu être rattachés à un quartier précis (le reste ne dispose pas de `CONSEIL DE QUARTIER` renseigné). Le pipeline utilise uniquement les lignes avec jointure réussie.
- **S3 au niveau arrondissement** : `parissetransforme` ne donne pas la localisation infra-arrondissement. Le score est donc identique pour les 4 quartiers d'un même arrondissement, modulé uniquement par leur surface.
- **Biais de signalement** : Les quartiers avec plus de résidents connectés génèrent plus de signalements (notamment les bobos du centre). Un quartier "silencieux" peut avoir un S1 artificiellement bon.

### Améliorations possibles
- Utiliser `geo_point_2d` (coordonnées GPS dans les CSV) pour une jointure spatiale précise via `GeoPandas`
- Pondérer le nombre de signalements par la **population du quartier** (données `RECENSEMENT_IRIS_POPULATION`)
- Ajouter un **score de démographie** (densité, part des jeunes, taux d'étrangers) depuis la table `silver_demographie`
- Intégrer une **dimension temporelle** (évolution du KPI par année) grâce à la colonne `ANNEE DECLARATION`

---

*Document généré automatiquement à partir du pipeline `gold_kpi_confort_urbain.py` — Projet Urban Data Explorer*
*Données : Paris Open Data — Millésime 2025*



### Voici le script complet de génération de ce KPI (données géographiques et mathématiques) :


In [ ]:
"""
gold_kpi_confort_urbain.py
Calcul du KPI de Confort Urbain pour les 80 quartiers de Paris.
"""

import os
import unicodedata

import numpy as np
import pandas as pd


def resolve_base_dir():
    cwd = os.getcwd()
    candidates = [cwd, os.path.dirname(cwd), os.path.dirname(os.path.dirname(cwd))]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'data')):
            return c
    if '__file__' in globals():
        return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    raise FileNotFoundError("Impossible de trouver la racine du projet (dossier 'data').")


BASE_DIR = resolve_base_dir()
BRONZE_COMMUN_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'commun')
BRONZE_CONFORT_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'confort')
BRONZE_IMMOBILIER_DIR = os.path.join(BASE_DIR, 'data', 'bronze', 'immobilier')
SILVER_CONFORT_DIR = os.path.join(BASE_DIR, 'data', 'silver', 'confort')
GOLD_DIR = os.path.join(BASE_DIR, 'data', 'gold')


def rang_centile(series: pd.Series) -> pd.Series:
    return series.rank(method='average', pct=True)


def col_find(df: pd.DataFrame, *keywords) -> str | None:
    for col in df.columns:
        if all(kw in col.lower() for kw in keywords):
            return col
    return None


def load_quartiers() -> pd.DataFrame:
    df = pd.read_csv(
        os.path.join(BRONZE_COMMUN_DIR, 'quartier_paris.csv'),
        sep=';', encoding='utf-8-sig'
    )
    rename = {
        'C_QU': 'code_quartier',
        'C_QUINSEE': 'code_insee_quartier',
        'L_QU': 'nom_quartier',
        'C_AR': 'arrondissement',
        'SURFACE': 'surface_m2',
        'Geometry X Y': 'centroide_xy',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    df['surface_m2'] = pd.to_numeric(df['surface_m2'], errors='coerce')
    df['arrondissement'] = pd.to_numeric(df['arrondissement'], errors='coerce').astype('Int64')
    df['code_postal_arr'] = df['code_insee_quartier'].astype(str).str[:5]
    df[['lat_q', 'lon_q']] = df['centroide_xy'].str.split(',', expand=True).astype(float)

    print(f"  Quartiers charges : {len(df)} lignes")
    return df[['code_quartier', 'code_insee_quartier', 'code_postal_arr',
               'nom_quartier', 'arrondissement', 'surface_m2', 'lat_q', 'lon_q']].copy()


TYPES_PROPRETE = ['Propreté', 'Voirie et espace public']


def compute_s1(df_rue: pd.DataFrame, quartiers: pd.DataFrame) -> pd.Series:
    print("  Calcul S1 - Proprete...")
    mask = df_rue['TYPE DECLARATION'].isin(TYPES_PROPRETE)
    df_prop = df_rue[mask].copy()
    agg = df_prop.groupby('code_insee_quartier').size().reset_index(name='n_proprete')

    df = quartiers.merge(agg, on='code_insee_quartier', how='left')
    df['n_proprete'] = df['n_proprete'].fillna(0)
    df['d_prop'] = df['n_proprete'] / df['surface_m2']
    df['rang_prop'] = rang_centile(df['d_prop'])
    df['S1'] = (10 * (1 - df['rang_prop'])).clip(0.01, 10)
    return df.set_index('code_quartier')['S1']


TYPES_INCIVIL = [
    'Objets abandonnés',
    'Graffitis, tags, affiches et autocollants',
    'Mobiliers urbains'
]

POIDS_INCIVIL = [
    (['infest', 'punaise'], 3.0),
    (['urine', 'panchement', 'épanchement'], 2.0),
    (['graffiti', 'mur', 'façade', 'rideau'], 1.5),
    (['déjection', 'canine'], 1.5),
    (['encombrant', 'carton', 'gravat'], 1.0),
    (['graffiti'], 1.3),
]


def get_poids_incivil(sous_type: str) -> float:
    if not isinstance(sous_type, str):
        return 1.0
    st_lower = sous_type.lower()
    for keywords, w in POIDS_INCIVIL:
        if all(kw in st_lower for kw in keywords):
            return w
    return 1.0


def compute_s2(df_rue: pd.DataFrame, quartiers: pd.DataFrame) -> pd.Series:
    print("  Calcul S2 - Incivilites...")
    mask = df_rue['TYPE DECLARATION'].isin(TYPES_INCIVIL)
    df_incivil = df_rue[mask].copy()
    df_incivil['poids'] = df_incivil['SOUS TYPE DECLARATION'].apply(get_poids_incivil)
    agg = df_incivil.groupby('code_insee_quartier')['poids'].sum().reset_index(name='I_incivil')

    df = quartiers.merge(agg, on='code_insee_quartier', how='left')
    df['I_incivil'] = df['I_incivil'].fillna(0)
    df['d_incivil'] = df['I_incivil'] / df['surface_m2']
    df['rang_incivil'] = rang_centile(df['d_incivil'])
    df['S2'] = (10 * (1 - df['rang_incivil'])).clip(0.01, 10)
    return df.set_index('code_quartier')['S2']


POIDS_AMENAGEMENT = {
    'Espace public / Nature en ville': 2.0,
    'Écoles et crèches': 1.8,
    'Propreté': 1.5,
    'Sécurité et Prévention': 1.3,
    'Sport': 1.0,
}
POIDS_AMENAGEMENT_DEFAULT = 0.5


def compute_s3(df_transf: pd.DataFrame, quartiers: pd.DataFrame) -> pd.Series:
    print("  Calcul S3 - Amenagement...")
    col_cat = col_find(df_transf, 'cat') or 'categorie'
    col_cp = col_find(df_transf, 'postal') or 'code_postal'

    df_transf = df_transf.copy()
    df_transf['poids'] = df_transf[col_cat].map(POIDS_AMENAGEMENT).fillna(POIDS_AMENAGEMENT_DEFAULT)
    df_transf[col_cp] = pd.to_numeric(df_transf[col_cp], errors='coerce')
    df_transf['arrondissement'] = (df_transf[col_cp] % 100).astype('Int64')

    agg_arr = df_transf.groupby('arrondissement')['poids'].sum().reset_index(name='I_am_arr')
    nb_q_par_arr = quartiers.groupby('arrondissement').size().reset_index(name='nb_quartiers')
    agg_arr = agg_arr.merge(nb_q_par_arr, on='arrondissement', how='left')
    agg_arr['I_am_quartier'] = agg_arr['I_am_arr'] / agg_arr['nb_quartiers']

    df = quartiers.merge(agg_arr[['arrondissement', 'I_am_quartier']], on='arrondissement', how='left')
    df['I_am_quartier'] = df['I_am_quartier'].fillna(0)
    df['d_am'] = df['I_am_quartier'] / df['surface_m2']
    df['rang_am'] = rang_centile(df['d_am'])
    df['S3'] = (10 * df['rang_am']).clip(0.01, 10)
    return df.set_index('code_quartier')['S3']


def compute_s4(df_loyers: pd.DataFrame, quartiers: pd.DataFrame) -> pd.Series:
    print("  Calcul S4 - Accessibilite logement...")

    def norm_col(s):
        return ''.join(c for c in unicodedata.normalize('NFD', str(s).lower())
                       if unicodedata.category(c) != 'Mn')

    def find_col(df, *frags):
        for col in df.columns:
            nc = norm_col(col)
            if all(norm_col(f) in nc for f in frags):
                return col
        return None

    df = df_loyers.copy()

    col_annee = find_col(df, 'ann')
    col_type = find_col(df, 'type', 'loc')
    col_pieces = find_col(df, 'pieces') or find_col(df, 'pi', 'ces')

    col_loyer = None
    for c in df.columns:
        nc = norm_col(c)
        if 'loyer' in nc and 'ference' in nc and 'major' not in nc and 'minor' not in nc:
            col_loyer = c
            break

    col_q = find_col(df, 'insee')

    print(f"    Colonnes -> annee:{col_annee} | type:{col_type} | pieces:{col_pieces} | loyer:{col_loyer} | insee:{col_q}")

    if not all([col_annee, col_type, col_pieces, col_loyer, col_q]):
        print("    ERREUR: colonnes loyers introuvables - S4 default a 5.0")
        s = pd.Series(5.0, index=quartiers['code_quartier'])
        s.index.name = 'code_quartier'
        return s

    df[col_annee] = pd.to_numeric(df[col_annee], errors='coerce')
    annee_max = df[col_annee].max()
    df = df[df[col_annee] == annee_max].copy()
    df = df[df[col_type].astype(str).str.lower().str.contains('non', na=False)]
    df = df[df[col_pieces].astype(str).str.contains('2', na=False)]

    df[col_loyer] = pd.to_numeric(df[col_loyer], errors='coerce')
    df[col_q] = pd.to_numeric(df[col_q], errors='coerce').astype('Int64')
    df = df.dropna(subset=[col_loyer]).loc[df[col_q].notna()]

    quartiers['code_insee_q_int'] = pd.to_numeric(quartiers['code_insee_quartier'], errors='coerce').astype('Int64')

    l_moy = df.groupby(col_q)[col_loyer].mean().reset_index()
    l_moy.columns = ['code_insee_q_int', 'L_moy']
    dq = quartiers.merge(l_moy, on='code_insee_q_int', how='left')

    med_arr = dq.groupby('arrondissement')['L_moy'].transform('median')
    dq['L_moy'] = dq['L_moy'].fillna(med_arr)
    dq['L_moy'] = dq['L_moy'].fillna(dq['L_moy'].median())

    L_med = dq['L_moy'].median()
    print(f"    Loyer median parisien ({annee_max}, non meuble, 2p) : {L_med:.2f} euros/m2")

    dq['delta'] = (dq['L_moy'] - L_med).abs() / L_med
    delta_max = dq['delta'].max()
    dq['S4'] = (10 * (1 - dq['delta'] / delta_max)).clip(0.01, 10)
    return dq.set_index('code_quartier')['S4']


def compute_s5(df_gares: pd.DataFrame, quartiers: pd.DataFrame) -> pd.Series:
    print("  Calcul S5 - Connectivite transport...")
    df_q = quartiers[['code_quartier', 'lat_q', 'lon_q']].copy()
    gares_coords = df_gares[['latitude', 'longitude']].dropna()

    RADIUS_KM = 0.8
    counts = []

    for _, row in df_q.iterrows():
        lat_q, lon_q = row['lat_q'], row['lon_q']
        dx = (gares_coords['longitude'] - lon_q) * 111.32 * np.cos(lat_q * np.pi / 180)
        dy = (gares_coords['latitude'] - lat_q) * 111.32
        dist_km = np.sqrt(dx**2 + dy**2)
        counts.append((dist_km <= RADIUS_KM).sum())

    df_q['nb_gares_800m'] = counts
    df_q['rang_transp'] = rang_centile(df_q['nb_gares_800m'])
    df_q['S5'] = (10 * df_q['rang_transp']).clip(0.01, 10)
    return df_q.set_index('code_quartier')['S5']


def load_dans_ma_rue(quartiers: pd.DataFrame) -> pd.DataFrame:
    print("  Chargement dans-ma-rue (peut prendre quelques secondes)...")
    cols = ['TYPE DECLARATION', 'SOUS TYPE DECLARATION',
            'ARRONDISSEMENT', 'CONSEIL DE QUARTIER',
            'CODE POSTAL', 'ANNEE DECLARATION']

    df = pd.read_csv(
        os.path.join(BRONZE_CONFORT_DIR, 'dans-ma-rue.csv'),
        sep=';', encoding='utf-8-sig',
        usecols=cols, on_bad_lines='skip'
    )

    print(f"    Lignes chargees : {len(df):,}")

    def normalize_name(s):
        if not isinstance(s, str):
            return ''
        return (s.upper()
                  .replace('-', ' ')
                  .replace('É', 'E').replace('È', 'E').replace('Ê', 'E')
                  .replace('À', 'A').replace('Â', 'A')
                  .replace('Ô', 'O').replace('Û', 'U').replace('Ü', 'U')
                  .replace('Î', 'I').replace('Ï', 'I')
                  .replace('Ç', 'C')
                  .strip())

    q_ref = quartiers.copy()
    q_ref['nom_norm'] = q_ref['nom_quartier'].apply(normalize_name)
    nom_to_code = dict(zip(q_ref['nom_norm'], q_ref['code_insee_quartier']))

    df['nom_cq_norm'] = df['CONSEIL DE QUARTIER'].apply(normalize_name)
    df['code_insee_quartier'] = df['nom_cq_norm'].map(nom_to_code)

    matched = df['code_insee_quartier'].notna().sum()
    total = len(df)
    print(f"    Jointure quartier : {matched:,}/{total:,} lignes matchees ({matched/total*100:.1f}%)")

    return df


def compute_gold_kpi():
    print("\n========================================")
    print("  GOLD KPI — Confort Urbain Paris")
    print("========================================\n")

    os.makedirs(GOLD_DIR, exist_ok=True)

    print("[1/6] Chargement referentiel quartiers...")
    quartiers = load_quartiers()

    print("[2/6] Chargement dans-ma-rue...")
    df_rue = load_dans_ma_rue(quartiers)
    df_rue_matched = df_rue[df_rue['code_insee_quartier'].notna()].copy()

    print("[3/6] Chargement parissetransforme...")
    df_transf = pd.read_csv(
        os.path.join(BRONZE_CONFORT_DIR, 'parissetransforme.csv'),
        sep=';', encoding='utf-8-sig', on_bad_lines='skip'
    )

    print("[4/6] Chargement loyers...")
    df_loyers = pd.read_csv(
        os.path.join(BRONZE_IMMOBILIER_DIR, 'logement-encadrement-des-loyers.csv'),
        sep=';', encoding='utf-8-sig', on_bad_lines='skip'
    )

    print("[5/6] Chargement gares (Silver)...")
    df_gares = pd.read_csv(
        os.path.join(SILVER_CONFORT_DIR, 'silver_gares.csv'),
        sep=';', encoding='utf-8-sig', on_bad_lines='skip'
    )

    print("\n[6/6] Calcul des scores par quartier...")
    s1 = compute_s1(df_rue_matched, quartiers)
    s2 = compute_s2(df_rue_matched, quartiers)
    s3 = compute_s3(df_transf, quartiers)
    s4 = compute_s4(df_loyers, quartiers)
    s5 = compute_s5(df_gares, quartiers)

    df_gold = quartiers.set_index('code_quartier').copy()
    df_gold['S1_proprete'] = s1
    df_gold['S2_incivilites'] = s2
    df_gold['S3_amenagement'] = s3
    df_gold['S4_accessibilite'] = s4
    df_gold['S5_transport'] = s5

    for col in ['S1_proprete', 'S2_incivilites', 'S3_amenagement', 'S4_accessibilite', 'S5_transport']】【：】【“】【        df_gold[col] = df_gold[col].fillna(df_gold[col].median())

    df_gold['KPI_confort_urbain'] = (
        df_gold['S1_proprete'] ** 0.30 *
        df_gold['S2_incivilites'] ** 0.25 *
        df_gold['S3_amenagement'] ** 0.15 *
        df_gold['S4_accessibilite'] ** 0.15 *
        df_gold['S5_transport'] ** 0.15
    ).round(3)

    df_gold = df_gold.reset_index().sort_values('KPI_confort_urbain', ascending=False)

    out_csv = os.path.join(GOLD_DIR, 'gold_kpi_confort_urbain.csv')
    df_gold.to_csv(out_csv, index=False, sep=';', encoding='utf-8-sig')

    print("\n========================================")
    print("  RESULTATS — Top 10 Quartiers (KPI)")
    print("========================================")
    cols_display = ['nom_quartier', 'arrondissement',
                    'S1_proprete', 'S2_incivilites',
                    'S3_amenagement', 'S4_accessibilite', 'S5_transport',
                    'KPI_confort_urbain']
    print(df_gold[cols_display].head(10).to_string(index=False))

    print("\n--- Bottom 10 ---")
    print(df_gold[cols_display].tail(10).to_string(index=False))

    print(f"\nCSV Gold         : {out_csv}")
    print("\nPIPELINE GOLD TERMINE AVEC SUCCES.")


if __name__ == "__main__":
    compute_gold_kpi()



  GOLD KPI — Confort Urbain Paris

[1/6] Chargement referentiel quartiers...
  Quartiers charges : 80 lignes
[2/6] Chargement dans-ma-rue...
  Chargement dans-ma-rue (peut prendre quelques secondes)...
    Lignes chargees : 1,062,299
    Jointure quartier : 111,344/1,062,299 lignes matchees (10.5%)
[3/6] Chargement parissetransforme...
[4/6] Chargement loyers...
[5/6] Chargement gares (Silver)...

[6/6] Calcul des scores par quartier...
  Calcul S1 - Proprete...
  Calcul S2 - Incivilites...
  Calcul S3 - Amenagement...
  Calcul S4 - Accessibilite logement...
    Colonnes -> annee:Année | type:Type de location | pieces:Nombre de pièces principales | loyer:Loyers de référence | insee:Numéro INSEE du quartier
    Loyer median parisien (2025, non meuble, 2p) : 29.15 euros/m2
  Calcul S5 - Connectivite transport...

  RESULTATS — Top 10 Quartiers (KPI)
      nom_quartier  arrondissement  S1_proprete  S2_incivilites  S3_amenagement  S4_accessibilite  S5_transport  KPI_confort_urbain
 Porte-